# FlashAttention：让 Attention 对硬件友好

> 在 inference 那一节，我们提到 FlashAttention 把 Q、K、V 切成小块在 SRAM 里算，避免把巨大的注意力矩阵反复写回显存。当时只用了「数学等价」一句话带过，没解释它凭什么能既快又精确。
>
> 这个附录从最基础的数值稳定性讲起，逐步推导 online softmax、tiling 分块，最后用 PyTorch 实现一个能和官方 `scaled_dot_product_attention` 对拍的教学版本。


标准 Attention 的公式只有一行：$O = \mathrm{softmax}(QK^\top / \sqrt{d}) V$。公式简洁，但直接实现会在两件事上吃亏。一是数值：$QK^\top$ 的元素可能很大，让 $e^{x}$ 溢出。二是显存：$N \times N$ 的注意力矩阵要先写到 HBM，再读回来做 softmax 和加权求和，读写量随序列长度平方增长。

FlashAttention 用两个核心 trick 同时解决这两件事。第一个是 online softmax：把「找 max」和「求和」两遍扫描合成一遍，过程中维护三个 running 变量 $(m, d, o)$。第二个是 tiling：把 Q、K、V 分块，每次只把一块 K、V 加载到 SRAM，算完一块更新一次 running 变量，中间结果从不写回 HBM。

整个过程不引入任何近似。最终输出和标准 Attention 在数学上完全相同，只在浮点误差上有 $10^{-6}$ 量级的差别。这正是它和 sparse attention、linear attention 的根本差异——后两者为了省计算主动放弃精确性，FlashAttention 只是为了省显存读写重新安排了计算顺序。


In [1]:
# 本附录只用 torch 和 numpy，不调任何高级 attention API
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(f"torch: {torch.__version__}")


torch: 2.8.0+rocm7.0.0.gitb2fb6885


## 1. GPU 的两层存储：HBM 与 SRAM

要理解 FlashAttention 在加速什么，先得看清 GPU 的存储层级。现代 GPU（比如 A100）有两层存储：

| 存储层 | 容量 | 带宽 | 角色 |
|:---|:---|:---|:---|
| HBM（高带宽显存） | 80 GB | ~2 TB/s | 主存，所有张量住在这里 |
| SRAM（片上缓存） | ~20 MB | ~19 TB/s | 每个 SM 私有，速度快近 10 倍 |

容量差 4000 倍，带宽差近 10 倍。这意味着：算一次矩阵乘法很快，但把数据从 HBM 搬到 SRAM 很慢。如果某个计算反复在 HBM 和 SRAM 之间搬运同一份数据，搬运时间会远超计算时间。

这种工作模式被称为 memory-bound。Attention 是典型的 memory-bound 算子。


### 标准 Attention 在搬运什么

标准 Attention 的 forward 分四步：

1. 计算 $S = QK^\top$，得到 $N \times N$ 的注意力分数矩阵，**写回 HBM**
2. 从 HBM 读回 $S$，算 row max，写回 HBM
3. 从 HBM 读回 $S$ 和 max，算 softmax 得到 $P$，**写回 HBM**
4. 从 HBM 读回 $P$ 和 $V$，算 $O = PV$，写回 HBM

每个 $N \times N$ 矩阵都要在 HBM 上来回读写多次。用具体数字感受一下规模。


In [2]:
# 标准注意力矩阵在不同序列长度下的显存占用（FP16，2 bytes/元素）
def attn_matrix_bytes(seq, head_dim=128, n_heads=32, bytes_per_elem=2):
    # 一次 forward 需要写回 HBM 的中间矩阵：S（attention scores）和 P（softmax 后）
    # 每个 head、每个 batch 都有一份
    elems = n_heads * seq * seq
    return elems * bytes_per_elem

def fmt(b):
    if b >= 1024**3:
        return f"{b/1024**3:.2f} GB"
    if b >= 1024**2:
        return f"{b/1024**2:.1f} MB"
    return f"{b/1024:.1f} KB"

print(f"{'序列长度':>8}  {'S/P 矩阵单份':>15}  {'4 次读写总量':>15}")
print("-" * 50)
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    single = attn_matrix_bytes(seq)
    # S 写 1 次、读 2 次（算 max、算 softmax），P 写 1 次、读 1 次
    traffic = single * 5
    print(f"{seq:>8d}  {fmt(single):>15}  {fmt(traffic):>15}")

print()
print("seq=32768 时，光搬运注意力矩阵就要 1 TB 量级的数据。")
print("A100 的 HBM 带宽 2 TB/s，光搬运就要 0.5 秒——比计算本身慢得多。")


    序列长度         S/P 矩阵单份          4 次读写总量
--------------------------------------------------
     512          16.0 MB          80.0 MB
    1024          64.0 MB         320.0 MB
    2048         256.0 MB          1.25 GB
    4096          1.00 GB          5.00 GB
    8192          4.00 GB         20.00 GB
   16384         16.00 GB         80.00 GB
   32768         64.00 GB        320.00 GB

seq=32768 时，光搬运注意力矩阵就要 1 TB 量级的数据。
A100 的 HBM 带宽 2 TB/s，光搬运就要 0.5 秒——比计算本身慢得多。


## 2. 数值稳定性：softmax 为什么会爆

进入 online softmax 之前，先解决一个更基础的问题：softmax 公式 $\frac{e^{x_i}}{\sum_j e^{x_j}}$ 直接算会爆。

指数函数 $e^x$ 增长极快。在 FP32 下，$e^{88}$ 左右就溢出到 inf。Attention 里 $QK^\top$ 的元素是两个 $d$ 维向量的内积，$d=128$ 时单个元素超过 100 很常见。


In [3]:
# 演示 exp 在 FP32 下的溢出阈值
import torch

x = torch.tensor([80.0, 85.0, 88.0, 90.0, 100.0])
print("x       exp(x)")
for xi in x:
    print(f"{xi.item():>6.1f}  {torch.exp(xi).item():>15.3e}")

print()
print("FP32 下 exp(89) 左右就接近 inf。")

# 模拟 attention score 的量级
d_head = 128
q = torch.randn(d_head)
k = torch.randn(d_head)
score = (q * k).sum() / math.sqrt(d_head)
print(f"随机初始化下 d=128 的 attention score 大约在 ±{abs(score.item()):.1f} 量级")
print("但训练后期 q、k 范数可能放大 5-10 倍，score 能到 80 以上。")


x       exp(x)
  80.0        5.541e+34
  85.0        8.223e+36
  88.0        1.652e+38
  90.0              inf
 100.0              inf

FP32 下 exp(89) 左右就接近 inf。
随机初始化下 d=128 的 attention score 大约在 ±1.8 量级
但训练后期 q、k 范数可能放大 5-10 倍，score 能到 80 以上。


### 2.1 stable softmax：减去 max

softmax 有一个关键性质：**对输入整体平移一个常数 $c$，输出不变**。

$$
\mathrm{softmax}(x - c)_i = \frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i} \cdot e^{-c}}{e^{-c} \sum_j e^{x_j}} = \mathrm{softmax}(x)_i
$$

$e^{-c}$ 在分子分母上同时出现，被约掉了。把 $c$ 取成 $x_\max$，那么最大的指数变成 $e^0 = 1$，所有指数都 $\le 1$，不再有溢出风险。

这就是 stable softmax：先扫一遍找 max，再扫一遍算 $\frac{e^{x_i - x_\max}}{\sum_j e^{x_j - x_\max}}$。


In [4]:
def naive_softmax(x):
    """直接实现，会溢出"""
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum()

def stable_softmax(x):
    """减去 max，数值稳定"""
    m = x.max()
    exp_x = torch.exp(x - m)
    return exp_x / exp_x.sum()

# 构造一个会让 naive 版爆炸的输入
x = torch.tensor([1.0, 2.0, 3.0]) * 50  # max 是 150
print("输入:", x.tolist())
print("naive softmax:", naive_softmax(x).tolist())
print("stable softmax:", stable_softmax(x).tolist())

print()
print("naive 版本里 exp(150) 溢出成 inf，最终结果变 nan。")
print("stable 版本里 max=150 被减掉，最大指数是 exp(0)=1，安全。")


输入: [50.0, 100.0, 150.0]
naive softmax: [0.0, nan, nan]
stable softmax: [3.783505853677006e-44, 1.9287498933537385e-22, 1.0]

naive 版本里 exp(150) 溢出成 inf，最终结果变 nan。
stable 版本里 max=150 被减掉，最大指数是 exp(0)=1，安全。


### 2.2 logsumexp trick

很多时候我们要算的是 $\log \sum_i e^{x_i}$（比如交叉熵损失里），它比 softmax 还容易爆。直接算 $\sum_i e^{x_i}$ 同样会溢出。

把 stable softmax 的思路反过来用：在 $\log$ 里把 max 提出来。

$$
\log \sum_i e^{x_i} = \log \left( e^{x_\max} \sum_i e^{x_i - x_\max} \right) = x_\max + \log \sum_i e^{x_i - x_\max}
$$

减掉 max 之后，最大的指数项是 $e^0 = 1$，求和至少是 1，$\log$ 不会取到 0。这就是 logsumexp trick，几乎所有深度学习框架的 `log_softmax` 都基于它。


In [5]:
def naive_logsumexp(x):
    """直接 log(sum(exp(x)))，会溢出"""
    return torch.log(torch.exp(x).sum())

def stable_logsumexp(x):
    """logsumexp trick"""
    m = x.max()
    return m + torch.log(torch.exp(x - m).sum())

x = torch.tensor([100.0, 101.0, 102.0])
print("输入:", x.tolist())
print("naive logsumexp:", naive_logsumexp(x).item())
print("stable logsumexp:", stable_logsumexp(x).item())
print("torch 官方:", torch.logsumexp(x, dim=0).item())

print()
print("三个应该完全相等（除了 naive 会爆 nan）。")
print("这个 trick 后面会在 online softmax 里反复出现——")
print("FlashAttention 的 running 变量本质上就是在维护 logsumexp 的流式版本。")


输入: [100.0, 101.0, 102.0]
naive logsumexp: inf
stable logsumexp: 102.40760803222656
torch 官方: 102.40760803222656

三个应该完全相等（除了 naive 会爆 nan）。
这个 trick 后面会在 online softmax 里反复出现——
FlashAttention 的 running 变量本质上就是在维护 logsumexp 的流式版本。


## 3. Online Softmax：把两遍扫描合成一遍

stable softmax 解决了溢出问题，但代价是要扫两遍数据：一遍找 $x_\max$，一遍算 $\sum_j e^{x_j - x_\max}$。对 GPU 来说，扫两遍意味着读两次 HBM。

FlashAttention 的第一个核心 trick 来自一个观察：**完全可以一边找 max 一边累加分母**，只要每次遇到更大的 max 时把之前累加的结果 rescale 一下。

这就是 online softmax。


### 3.1 两遍算法 vs 单遍算法

经典 stable softmax 的两遍流程：

```
第一遍：扫 x_1, ..., x_N，记录 m = max
第二遍：扫 x_1, ..., x_N，累加 d = sum exp(x_i - m)
返回：softmax_i = exp(x_i - m) / d
```

online softmax 改成单遍：

```
扫描 x_1, ..., x_N，同时维护：
  m_k = max(x_1, ..., x_k)            # running max
  d_k = sum_{j=1}^k exp(x_j - m_k)    # running 分母（用当前 max shift 过）

每来一个新的 x_{k+1}：
  1. 更新 max: m_{k+1} = max(m_k, x_{k+1})
  2. rescale 旧累加: d_{k+1} = d_k * exp(m_k - m_{k+1}) + exp(x_{k+1} - m_{k+1})
```

关键在第二步：当 max 从 $m_k$ 涨到 $m_{k+1}$ 时，之前所有 $\exp(x_j - m_k)$ 都要重新以 $m_{k+1}$ 为基准。但 $\exp(x_j - m_{k+1}) = \exp(x_j - m_k) \cdot \exp(m_k - m_{k+1})$，所以只要给 $d_k$ 整体乘一个 $\exp(m_k - m_{k+1})$ 就完成了 rescale。

当 max 没变（$m_k = m_{k+1}$）时，$\exp(m_k - m_{k+1}) = \exp(0) = 1$，旧累加不动，直接加上新项。


### 3.2 手算一遍

用 $x = [3, 1, 2]$ 演示。最终的 stable softmax 是 $\frac{[e^3, e^1, e^2]}{e^3 + e^1 + e^2}$，max=3。

按 online 流程一步步走：

| 步骤 | $x_{k+1}$ | 新 max $m$ | rescale 因子 $e^{m_\text{old} - m_\text{new}}$ | 新分母 $d$ |
|:---|:---|:---|:---|:---|
| 初始 | - | $-\infty$ | - | 0 |
| 看到 $x_1=3$ | 3 | $\max(-\infty, 3) = 3$ | $e^{-\infty - 3} = 0$ | $0 \cdot 0 + e^{3-3} = 1$ |
| 看到 $x_2=1$ | 1 | $\max(3, 1) = 3$ | $e^{3-3} = 1$ | $1 \cdot 1 + e^{1-3} = 1 + 0.135 = 1.135$ |
| 看到 $x_3=2$ | 2 | $\max(3, 2) = 3$ | $e^{3-3} = 1$ | $1.135 \cdot 1 + e^{2-3} = 1.135 + 0.368 = 1.503$ |

最终 $m=3$、$d=1.503$。验算一下：$e^3 + e^1 + e^2 = 20.09 + 2.72 + 7.39 = 30.20$，而 $d \cdot e^m = 1.503 \cdot e^3 = 1.503 \cdot 20.09 = 30.20$。两者相等，说明 online 算出来的 $(m, d)$ 和经典算法完全一致。

注意整个过程只扫了一遍 $x$，且每一步都只用到当前元素和三个 running 变量。


In [6]:
def online_softmax(x):
    """单遍 softmax，维护 running (m, d)"""
    m = float('-inf')   # running max
    d = 0.0             # running 分母 sum exp(x_i - m)

    for xi in x:
        m_new = max(m, xi.item())
        # rescale 旧累加，加上新项
        d = d * math.exp(m - m_new) + math.exp(xi.item() - m_new)
        m = m_new

    # 第二遍：用最终的 (m, d) 算每个位置的 softmax
    # 这一步 FlashAttention 会通过 tiling 融合进 forward
    return torch.tensor([math.exp(xi.item() - m) / d for xi in x])

x = torch.tensor([3.0, 1.0, 2.0])
print("输入:", x.tolist())
print("online softmax:    ", online_softmax(x).tolist())
print("stable softmax:    ", stable_softmax(x).tolist())
print("torch 官方 softmax:", torch.softmax(x, dim=0).tolist())

print()
print("三者数值完全相同——online 流程没有引入任何近似。")


输入: [3.0, 1.0, 2.0]
online softmax:     [0.6652409434318542, 0.09003057330846786, 0.2447284758090973]
stable softmax:     [0.6652408838272095, 0.09003056585788727, 0.2447284609079361]
torch 官方 softmax: [0.6652408838272095, 0.09003056585788727, 0.2447284460067749]

三者数值完全相同——online 流程没有引入任何近似。


## 4. 从 softmax 到 Attention 输出

attention 的最终输出不是 softmax 本身，而是 $o = \sum_i p_i v_i$，其中 $p_i$ 是 softmax 权重，$v_i$ 是 value 向量。把 $p_i$ 展开：

$$
o = \sum_i \frac{e^{x_i}}{\sum_j e^{x_j}} v_i = \frac{\sum_i e^{x_i} v_i}{\sum_i e^{x_i}}
$$

分子分母同乘 $e^{-x_\max}$ 抵消溢出：

$$
o = \frac{\sum_i e^{x_i - x_\max} v_i}{\sum_i e^{x_i - x_\max}}
$$

这里 $x_i = q \cdot k_i / \sqrt{d}$ 是 query 和第 $i$ 个 key 的点积，$v_i$ 是第 $i$ 个 value。除了 $(m, d)$ 之外，再维护一个 running 输出 $o_k = \sum_{i=1}^k e^{x_i - m_k} v_i$。

更新规则和 $d$ 完全对称，因为本质都是「$e^{x_i - m}$ 的加权和」：

$$
o_{k+1} = o_k \cdot e^{m_k - m_{k+1}} + e^{x_{k+1} - m_{k+1}} v_{k+1}
$$

扫完所有 $N$ 个 key 之后，真正的 attention 输出就是 $o_N / d_N$。


### 4.1 手算 attention 输出

用一个具体的 query 和三个 key-value 对，验证 online 流程的正确性。设 $d=1$ 简化数字（实际 $d=64$ 或 $128$）：

- $q = 1$，三个 key $k = [3, 1, 2]$，对应 value $v = [10, 20, 30]$
- attention score $x_i = q \cdot k_i = [3, 1, 2]$（省略 $\sqrt{d}$ 因为 $d=1$）
- 真实输出 $o = \frac{e^3 \cdot 10 + e^1 \cdot 20 + e^2 \cdot 30}{e^3 + e^1 + e^2}$

按 online 流程走：

| 步骤 | $x_{k+1}$ | $v_{k+1}$ | $m$ | rescale | $d$ | $o$ |
|:---|:---|:---|:---|:---|:---|:---|
| 初始 | - | - | $-\infty$ | - | 0 | 0 |
| $k=1$ | 3 | 10 | 3 | 0 | $e^0 = 1$ | $e^0 \cdot 10 = 10$ |
| $k=2$ | 1 | 20 | 3 | 1 | $1 + e^{-2} = 1.135$ | $10 + e^{-2} \cdot 20 = 10 + 2.71 = 12.71$ |
| $k=3$ | 2 | 30 | 3 | 1 | $1.135 + e^{-1} = 1.503$ | $12.71 + e^{-1} \cdot 30 = 12.71 + 11.04 = 23.75$ |

最终 $o / d = 23.75 / 1.503 = 15.80$。直接按定义算 $\frac{20.09 \cdot 10 + 2.72 \cdot 20 + 7.39 \cdot 30}{30.20} = \frac{200.9 + 54.4 + 221.7}{30.20} = \frac{477.0}{30.20} = 15.80$。两者相等。


In [7]:
def online_attention(q, K, V):
    """
    单遍 attention，每个 query 独立处理

    参数:
        q: 一个 query 向量，shape [d]
        K: 所有 key，shape [N, d]
        V: 所有 value，shape [N, d]
    返回:
        attention 输出，shape [d]
    """
    d_head = q.shape[0]
    N = K.shape[0]

    m = float('-inf')       # running max
    d = 0.0                 # running 分母
    o = torch.zeros_like(q) # running 输出

    for i in range(N):
        # 当前 key 与 query 的点积（除以 sqrt(d) 保证梯度稳定）
        x_i = torch.dot(q, K[i]).item() / math.sqrt(d_head)
        v_i = V[i]

        m_new = max(m, x_i)
        rescale = math.exp(m - m_new)

        d = d * rescale + math.exp(x_i - m_new)
        o = o * rescale + math.exp(x_i - m_new) * v_i
        m = m_new

    return o / d

# 手算例子：d=1，q=1，k=[3,1,2]，v=[10,20,30]
q = torch.tensor([1.0])
K = torch.tensor([[3.0], [1.0], [2.0]])
V = torch.tensor([[10.0], [20.0], [30.0]])
print("online attention 输出:", online_attention(q, K, V).item())
print("手算结果: 15.80")

# 对照标准实现
def standard_attention(q, K, V):
    d_head = q.shape[0]
    scores = (K @ q) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=0)
    return weights @ V

print("standard attention 输出:", standard_attention(q, K, V).item())


online attention 输出: 15.79487419128418
手算结果: 15.80
standard attention 输出: 15.794873237609863


## 5. Tiling：把单遍算法搬上 GPU

到这一步，我们已经能在 CPU 上用单遍循环算 attention。但 GPU 不是这样工作的——GPU 要靠大规模并行，一个一个 token 循环是最差的形式。

FlashAttention 的第二个核心 trick 是 tiling：把 Q、K、V 切成块，让每块刚好能塞进 SRAM，循环在块级别进行。

### 5.1 分块结构

设序列长度 $N$、head 维度 $d$、block size $B$（通常取 64 或 128）。把序列切成 $N / B$ 块：

- 外层循环遍历 Q 的块：$Q_1, Q_2, \dots, Q_{N/B}$，每块 shape $[B, d]$
- 内层循环遍历 K、V 的块：$K_1, K_2, \dots, K_{N/B}$，每块 shape $[B, d]$

对每一个 Q 块，独立维护一组 running 变量 $(m, d, o)$，shape 分别是 $[B]$、$[B]$、$[B, d]$。在内层循环里，每读入一对 $(K_j, V_j)$ 块就更新一次。

```
for each Q block i (outer loop, parallelizable across blocks):
    m_i = -inf, d_i = 0, o_i = 0   # shape [B], [B], [B, d]
    for each K, V block j (inner loop):
        load K_j, V_j from HBM to SRAM
        S = Q_i @ K_j^T            # [B, B]，只在 SRAM 里存在
        m_block = S.max(dim=-1)    # [B]
        m_new = max(m_i, m_block)
        P = exp(S - m_new[:, None])
        d_i = d_i * exp(m_i - m_new) + P.sum(dim=-1)
        o_i = o_i * exp(m_i - m_new)[:, None] + P @ V_j
        m_i = m_new
    write o_i / d_i back to HBM   # 只写最终结果
```

### 5.2 为什么 tiling 不破坏正确性

online softmax 的更新规则是**可结合的**：先在块内累加，再用块级 max rescale，和一次性扫完所有元素结果相同。原因是更新规则只依赖三个量 $(m, d, o)$，块的边界对最终结果没有影响。

直觉上：把 $N$ 个 key 分成两半先各自算 $(m_1, d_1, o_1)$ 和 $(m_2, d_2, o_2)$，再用全局 max $m = \max(m_1, m_2)$ rescale 合并，等价于一次性扫完。这就是块级 online softmax 的数学保证。


### 5.3 block size 怎么选

block size $B$ 决定 SRAM 占用。每块需要存 $Q_i$ ($B \times d$)、$K_j$ ($B \times d$)、$V_j$ ($B \times d$)、中间 $S$ 和 $P$（各 $B \times B$）。SRAM 总占用大致是 $3 B d + 2 B^2$ 个元素。

A100 的 SRAM 约 19 MB（多个 SM 分摊），$d = 128$ 时 $B = 128$ 是常见选择，占大约 $3 \cdot 128 \cdot 128 + 2 \cdot 128^2 = 81920$ 个 FP16 元素，约 160 KB。每个 SM 同时处理多个 block，整体刚好塞满 SRAM。

$B$ 太小：循环次数多，kernel launch 开销大。$B$ 太大：SRAM 装不下，会被迫退回 HBM。在生产实现里，$B$ 通常由 GPU 型号和 head 维度共同决定，不需要用户调。


## 6. 为什么是 exact，不是近似

这一节澄清一个常见误解：FlashAttention 不牺牲精度。

sparse attention（比如 Longformer、Big Bird）主动跳过一部分 query-key 对，认为「远距离的注意力不重要」。这是**结构上的近似**——输出和 dense attention 数学上不同。

linear attention（Performer、Linear Transformer）用 kernel trick 把 $QK^\top$ 替换成可分解的核函数，把 $O(N^2)$ 降到 $O(N)$。这也是**数学上的近似**。

FlashAttention 不一样。它只是**重新安排了计算顺序**：

- dense 版本：先算完整 $S$，再 softmax，再乘 $V$
- flash 版本：分块算 $S$ 的一小块，立刻 softmax，立刻乘 $V$ 的一小块，running 累加

两者的数学运算完全等价，只是中间结果出现的位置不同。FlashAttention 永远不写完整 $S$ 到 HBM，但每个 $S_{ij}$ 都参与了计算。最终输出的差距只有浮点误差量级（$10^{-6}$ 左右）。

这就是为什么 FlashAttention 可以无缝替换标准 attention——训练曲线、模型精度、收敛行为都不变。


## 7. 简化 PyTorch 实现

下面用纯 PyTorch 实现一个教学版 FlashAttention。**不追求性能**（Python 循环比 CUDA 慢得多），只追求数学正确——让读者看清 online softmax + tiling 的每一步在做什么。

实现要点：

- 单 head、batch=1 简化（多 head 只是外层循环）
- 外层 Q 块循环，内层 K/V 块循环
- 用 `torch.einsum` 或 `@` 处理块级矩阵乘法
- 显式维护 $(m, d, o)$ 三个 running 变量


In [8]:
def flash_attention_forward(Q, K, V, block_size=4, causal=False):
    """
    教学版 FlashAttention forward，单 head，batch=1

    参数:
        Q: shape [N, d]
        K: shape [N, d]
        V: shape [N, d]
        block_size: Q/K/V 分块大小
        causal: 是否因果 mask（下三角）
    返回:
        O: shape [N, d]
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # 输出和 running 变量都在 HBM 上（教学版简化）
    # 真实 FlashAttention 把它们放在 SRAM
    O = torch.zeros(N, d)

    # 外层循环：每个 Q 块独立处理
    n_blocks = (N + block_size - 1) // block_size
    for i in range(n_blocks):
        q_start = i * block_size
        q_end = min(q_start + block_size, N)
        Qi = Q[q_start:q_end] * scale  # [B_q, d]，提前乘 scale

        # 这个 Q 块的 running 变量
        B_q = q_end - q_start
        m_i = torch.full((B_q,), float('-inf'))
        d_i = torch.zeros(B_q)
        o_i = torch.zeros(B_q, d)

        # 内层循环：遍历所有 K/V 块
        for j in range(n_blocks):
            k_start = j * block_size
            k_end = min(k_start + block_size, N)
            Kj = K[k_start:k_end]  # [B_k, d]
            Vj = V[k_start:k_end]  # [B_k, d]

            # 算这一块的 attention score（只在 SRAM 里存在）
            S = Qi @ Kj.transpose(-2, -1)  # [B_q, B_k]

            # 因果 mask：query 位置 < key 位置的地方置 -inf
            if causal:
                q_idx = torch.arange(q_start, q_end)[:, None]
                k_idx = torch.arange(k_start, k_end)[None, :]
                mask = k_idx > q_idx
                S = S.masked_fill(mask, float('-inf'))

            # online softmax 更新
            m_block = S.max(dim=-1).values      # [B_q]
            m_new = torch.maximum(m_i, m_block)

            # 旧累加 rescale
            alpha = torch.exp(m_i - m_new)      # [B_q]
            # 新块的 exp（减去新 max 保证数值稳定）
            P = torch.exp(S - m_new[:, None])   # [B_q, B_k]

            d_i = d_i * alpha + P.sum(dim=-1)
            o_i = o_i * alpha[:, None] + P @ Vj
            m_i = m_new

        # 这个 Q 块的最终输出
        O[q_start:q_end] = o_i / d_i[:, None]

    return O

# 小例子：seq=8, d=4，手算友好
torch.manual_seed(42)
N, d = 8, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

O_flash = flash_attention_forward(Q, K, V, block_size=4)
print("FlashAttention 输出 (前 3 行):")
print(O_flash[:3])


FlashAttention 输出 (前 3 行):
tensor([[ 0.2733, -0.0495, -0.3578,  0.3780],
        [ 0.3305, -0.0531, -0.5517,  0.6123],
        [ 0.2625, -0.1712, -0.8024, -0.1736]])


## 8. 对拍验证

光实现不够，要证明它和标准 attention 数值相同。对照两种参考实现：

1. 手写的标准 attention（显式构造 $N \times N$ 矩阵）
2. PyTorch 官方的 `scaled_dot_product_attention`（底层会自动选 FlashAttention 后端）


In [9]:
def standard_attention(Q, K, V, causal=False):
    """标准 attention，显式构造 N×N 矩阵"""
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    S = (Q @ K.T) * scale  # [N, N]
    if causal:
        mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
        S = S.masked_fill(mask, float('-inf'))
    P = torch.softmax(S, dim=-1)
    return P @ V

# 非因果场景
O_std = standard_attention(Q, K, V, causal=False)
O_sdpa = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=False
)[0]

print("=== 非因果场景 ===")
print(f"Flash vs 标准实现 最大差: {(O_flash - O_std).abs().max().item():.2e}")
print(f"Flash vs SDPA     最大差: {(O_flash - O_sdpa).abs().max().item():.2e}")
print(f"标准  vs SDPA     最大差: {(O_std - O_sdpa).abs().max().item():.2e}")

# 因果场景
O_flash_causal = flash_attention_forward(Q, K, V, block_size=4, causal=True)
O_std_causal = standard_attention(Q, K, V, causal=True)
O_sdpa_causal = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=True
)[0]

print()
print("=== 因果场景 ===")
print(f"Flash vs 标准实现 最大差: {(O_flash_causal - O_std_causal).abs().max().item():.2e}")
print(f"Flash vs SDPA     最大差: {(O_flash_causal - O_sdpa_causal).abs().max().item():.2e}")

print()
print("所有差异都在 1e-6 量级——浮点误差范围内，FlashAttention 数学等价。")


=== 非因果场景 ===
Flash vs 标准实现 最大差: 8.94e-08
Flash vs SDPA     最大差: 1.79e-07
标准  vs SDPA     最大差: 1.19e-07

=== 因果场景 ===
Flash vs 标准实现 最大差: 1.19e-07
Flash vs SDPA     最大差: 1.19e-07

所有差异都在 1e-6 量级——浮点误差范围内，FlashAttention 数学等价。


In [10]:
# 更大的测试：seq=256，d=64，看长序列下也保持精度
torch.manual_seed(0)
N, d = 256, 64
Q_big = torch.randn(N, d)
K_big = torch.randn(N, d)
V_big = torch.randn(N, d)

O_flash_big = flash_attention_forward(Q_big, K_big, V_big, block_size=64)
O_sdpa_big = F.scaled_dot_product_attention(
    Q_big[None], K_big[None], V_big[None]
)[0]

max_diff = (O_flash_big - O_sdpa_big).abs().max().item()
mean_diff = (O_flash_big - O_sdpa_big).abs().mean().item()
print(f"seq=256, d=64:")
print(f"  最大差:  {max_diff:.2e}")
print(f"  平均差:  {mean_diff:.2e}")
print(f"  通过 1e-5 阈值: {max_diff < 1e-5}")

# 顺便看 block size 对结果的影响（应该完全不影响）
print()
print("block size 对数值正确性的影响（应全部 < 1e-5）：")
for bs in [16, 32, 64, 128]:
    O_bs = flash_attention_forward(Q_big, K_big, V_big, block_size=bs)
    diff = (O_bs - O_sdpa_big).abs().max().item()
    print(f"  block_size={bs:>3d}: 最大差 {diff:.2e}")


seq=256, d=64:
  最大差:  7.75e-07
  平均差:  2.82e-08
  通过 1e-5 阈值: True

block size 对数值正确性的影响（应全部 < 1e-5）：


  block_size= 16: 最大差 4.77e-07


  block_size= 32: 最大差 5.96e-07


  block_size= 64: 最大差 7.75e-07
  block_size=128: 最大差 1.01e-06


## 9. 在 MoE 场景下的特别考虑

MoE（Mixture of Experts）模型的 attention 部分和 dense 模型完全一样——每个 expert 内部的 self-attention 仍然是标准的因果多头 attention，FlashAttention 的用法不变。

但 MoE 引入了一个新问题：**expert routing 之后会产生大量小 batch 的 attention 调用**。

### 9.1 MoE 的 attention 数据流

设 batch size $B$、seq len $S$、expert 数 $E$、top-k 路由 $k$。每个 token 被路由到 $k$ 个 expert，平均每个 expert 收到 $B \cdot S \cdot k / E$ 个 token。

- $B=1, S=4096, k=2, E=8$：每个 expert 平均收到 1024 个 token
- $B=1, S=4096, k=1, E=64$：每个 expert 平均收到 64 个 token

这些 token 来自原序列的不同位置，**不能直接拼成连续序列做 attention**——它们的因果关系是相对于原序列的，不是相对于 expert 内部的。

### 9.2 两种处理方式

第一种是 **padding**：每个 expert 内部维护一个 $B \times S$ 的张量，被路由到该 expert 的 token 留下，其他位置填 0。attention 仍然在 $B \times S$ 上算，但有效 token 只有 $B \cdot S \cdot k / E$ 个。这造成大量浪费——算力和显存都按 $B \times S$ 计，但只有 $k/E$ 的比例真正参与。

第二种是 **grouped GEMM / grouped attention**：把每个 expert 的不连续 token 收集到一起，做成变长序列，让 attention kernel 直接处理变长输入。FlashAttention 的变长版本（varlen）支持这种模式，避免 padding。

| 方案 | 计算/显存浪费 | 实现复杂度 |
|:---|:---|:---|
| Padding | $E/k$ 倍 | 简单 |
| Grouped (varlen) | 接近 0 | 需要 custom kernel |

### 9.3 对 FlashAttention 的影响

FlashAttention 算法本身在 MoE 下没有特殊处理——只要能给它正确的 $(Q, K, V)$ 三元组，它就按标准流程算。差异在于**怎么组织输入**：

- dense 模型：一个 $[B, N\_\text{heads}, S, d]$ 的大张量，一次调用
- MoE padding：每个 expert 一个 $[B, N\_\text{heads}, S, d]$ 张量，$E$ 次调用，每次大半是 padding
- MoE varlen：把所有 expert 的有效 token 拼成一个长 1D 张量，配上每个 expert 的 cu_seqlens（累积长度），一次调用

varlen 是 MoE + FlashAttention 的推荐方式。Megatron-LM、vLLM、SGLang 等推理框架都基于 varlen 实现 grouped attention。


In [11]:
# 量化 MoE 下 padding 的浪费
def moe_padding_waste(B, S, E, k, n_heads, d):
    """计算 MoE 在 padding 模式下的有效算力占比"""
    # 每个 expert 收到的平均 token 数
    avg_tokens_per_expert = B * S * k / E
    # padding 模式下每个 expert 仍然在 [B, S] 上算
    actual_tokens = B * S
    # 有效占比
    eff_ratio = avg_tokens_per_expert / actual_tokens
    # 总 FLOPs（只看 attention score 的 matmul）
    flops_padding = E * B * n_heads * S * S * d * 2
    flops_useful = E * avg_tokens_per_expert * n_heads * S * d * 2
    return eff_ratio, flops_padding, flops_useful

print(f"{'配置':>30}  {'有效 token 占比':>15}  {'浪费倍数':>10}")
print("-" * 65)
configs = [
    ("dense (E=1, k=1)", 1, 4096, 1, 1, 32, 128),
    ("MoE E=8, k=2", 1, 4096, 8, 2, 32, 128),
    ("MoE E=64, k=1", 1, 4096, 64, 1, 32, 128),
    ("MoE E=64, k=2", 1, 4096, 64, 2, 32, 128),
    ("MoE E=128, k=2", 1, 4096, 128, 2, 32, 128),
]
for name, B, S, E, k, nh, d in configs:
    eff, _, _ = moe_padding_waste(B, S, E, k, nh, d)
    waste = 1 / eff if eff > 0 else float('inf')
    print(f"{name:>30}  {eff:>14.1%}  {waste:>9.1f}x")

print()
print("E=128, k=2 时，padding 模式有效算力只有 1.6%，浪费 64 倍。")
print("varlen 模式直接绕开这个问题。")


                            配置      有效 token 占比        浪费倍数
-----------------------------------------------------------------
              dense (E=1, k=1)          100.0%        1.0x
                  MoE E=8, k=2           25.0%        4.0x
                 MoE E=64, k=1            1.6%       64.0x
                 MoE E=64, k=2            3.1%       32.0x
                MoE E=128, k=2            1.6%       64.0x

E=128, k=2 时，padding 模式有效算力只有 1.6%，浪费 64 倍。
varlen 模式直接绕开这个问题。


## 附录：FlashAttention 的演进

### A7.1 FlashAttention-2：更少的非 matmul FLOPs

FlashAttention-1 的 forward 已经很快，但反向传播和 work partitioning 还有改进空间。FlashAttention-2（Dao 2023）针对三点优化。

第一，**减少非 matmul FLOPs**。GPU 上 matmul（tensor core）的吞吐远高于其他运算（比如 rescale、softmax）。FA-1 在 inner loop 里有一些非 matmul 操作，FA-2 把它们重新安排，让 tensor core 的利用率更高。

第二，**更好的并行化**。FA-1 主要在 batch 和 head 维度并行，序列长但 batch 小的场景下 GPU 利用率低。FA-2 把序列维度也拆开并行——不同的 Q 块分到不同 SM 上，长序列场景下吞吐显著提升。

第三，**work partitioning**。FA-1 的 inner/outer loop 划分让 GPU 的 warp 之间频繁同步。FA-2 重新设计让每个 warp 独立处理一块，减少同步开销。

实测 FA-2 比 FA-1 快约 2 倍，达到 A100 上 matmul 理论吞吐的 50-70%。

### A7.2 FlashAttention-3：Hopper 架构与 FP8

FlashAttention-3（Shah et al. 2024）专门为 Hopper 架构（H100）设计，利用三个新特性。

第一，**WGMMA（warp-group matrix multiply accumulate）**。Hopper 引入的异步 tensor core 指令，matmul 可以和其他运算 overlap。FA-3 把 softmax 和 rescale 安排在 matmul 的间隙里执行，几乎完全隐藏非 matmul 开销。

第二，**TMA（tensor memory accelerator）**。Hopper 的硬件 DMA，可以异步在 HBM 和 SRAM 之间搬数据，不占用 SM。FA-3 用 TMA 做双层 buffering——下一块 K/V 在算当前块时就已经预取。

第三，**FP8 tensor core**。Hopper 支持 FP8（E4M3 / E5M2）的 tensor core matmul，吞吐是 FP16 的两倍。FA-3 在 forward 用 FP8 算 $QK^\top$ 和 $PV$，但 softmax 部分仍用 FP32 累加保证数值稳定。

实测 FA-3 在 H100 上达到 1.5-2 PFLOPs（FP8），接近硬件理论上限的 75%。

### A7.3 Triton 实现导读

OpenAI 的 Triton 是 Python 写 GPU kernel 的高级语言，比 CUDA 更易读。FlashAttention 官方维护了一份 Triton 实现，是学习 tiling kernel 的最佳参考。

源码位于 [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) 仓库的 `flash_attn/flash_attn_triton.py`。核心结构是一个 `@triton.jit` 装饰的函数，外层 Q 块循环用 `tl.range`，内层 K/V 块循环也是 `tl.range`，每个块用 `tl.load` 从 HBM 读、用 `tl.store` 写回。online softmax 的 $(m, d, o)$ 更新直接翻译成几行 Triton 代码，和本附录实现的 PyTorch 版逻辑完全对应。

阅读建议：先看本附录的 PyTorch 实现建立直觉，再对照 Triton 版理解 GPU 编程细节（shared memory、warp 同步、block 指针推进）。Triton 官方教程也有一份 [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) 的简化版本，比 Dao 官方版更易读。


### 参考文献

- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018（online softmax 的原始论文）
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), 2019


## 小结

- [ ] GPU 有两层存储：HBM 大但慢（~2 TB/s），SRAM 小但快（~19 TB/s），attention 是 memory-bound 的代表
- [ ] 标准 attention 把 $N \times N$ 中间矩阵写回 HBM 4-5 次，序列一长搬运量远超计算量
- [ ] softmax 直接算会溢出（FP32 下 $e^{88}$ 接近 inf），减去 max 的 stable softmax 是基础防线
- [ ] logsumexp trick 把 $\log \sum e^{x_i}$ 变成 $x_\max + \log \sum e^{x_i - x_\max}$，避免取 $\log(0)$ 和 $\exp$ 溢出
- [ ] online softmax 把「找 max」和「求和」合成一遍，靠 running $(m, d)$ 和 $e^{m_\text{old} - m_\text{new}}$ rescale 完成等价计算
- [ ] attention 输出 $o = \sum p_i v_i$ 同样能用 running $o_k$ 单遍累加，规则和 $d$ 对称
- [ ] tiling 把 Q、K、V 切块塞进 SRAM，块级 online softmax 因为更新规则可结合而保持正确
- [ ] FlashAttention 是 exact 不是近似——只是重排计算顺序，和 sparse / linear attention 有本质区别
- [ ] MoE 场景下 attention 本身不变，但 expert routing 产生大量小 batch，padding 模式浪费严重，varlen 模式更优
- [ ] FlashAttention-2 优化 work partitioning，FlashAttention-3 利用 Hopper 的 WGMMA / TMA / FP8


## 作业

下面三道作业帮你巩固 online softmax 和 attention 的核心更新规则。可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

**作业 1：实现 running max 更新**

下面是一个简化版的 online softmax 框架，缺少 running max 和 rescale 部分。补全 `m_new`、`rescale`、`d` 三行，使输出和 `torch.softmax` 完全一致。

```python
def online_softmax_homework(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        xi = xi.item()
        # TODO: 补全三行
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        m = m_new
    return torch.tensor([math.exp(xi - m) / d for xi in x])

x = torch.tensor([1.0, 5.0, 3.0, 7.0, 2.0])
result = online_softmax_homework(x)
expected = torch.softmax(x, dim=0)
assert torch.allclose(result, expected, atol=1e-6), f"差值过大: {(result - expected).abs().max()}"
print("作业 1 通过：online softmax 实现正确")
```

小提示：参考第 3.1 节的更新规则——新 max 是旧 max 和当前值的较大者，rescale 因子是 $\exp(m_\text{old} - m_\text{new})$，分母是「旧分母乘 rescale」加上「新项 $\exp(x_i - m_\text{new})$」。

**作业 2：验证 online attention 的数值等价**

补全下面的函数，让它对随机输入 $(Q, K, V)$ 的输出和 `F.scaled_dot_product_attention` 差异小于 $10^{-5}$。

```python
def online_attention_homework(q, K, V):
    """单 query 版 attention"""
    d_head = q.shape[0]
    m = float('-inf')
    d = 0.0
    o = torch.zeros_like(q)

    for i in range(K.shape[0]):
        x_i = (q * K[i]).sum().item() / math.sqrt(d_head)
        v_i = V[i]
        # TODO: 补全 m_new、rescale、d、o 四步更新
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        # 4. o = ?
        m = m_new

    return o / d

torch.manual_seed(0)
d = 8
N = 16
q = torch.randn(d)
K = torch.randn(N, d)
V = torch.randn(N, d)
out = online_attention_homework(q, K, V)
ref = F.scaled_dot_product_attention(q[None, None], K[None], V[None])[0, 0]
assert torch.allclose(out, ref, atol=1e-5), f"差值: {(out - ref).abs().max()}"
print("作业 2 通过：online attention 数值等价")
```

小提示：$o$ 的更新规则和 $d$ 完全对称——都是「旧值乘 rescale 因子」加上「$\exp(x_i - m_\text{new})$ 乘以对应的 $v_i$」。参考第 4 节的公式。

**作业 3：估 MoE padding 浪费**

一个 MoE 模型，$E=64$ 个 expert，top-1 路由（$k=1$），序列长度 $S=8192$，batch $B=4$。Padding 模式下每个 expert 的 attention 调用仍然处理 $[B, S]$ 的张量。

(1) 每个 expert 平均收到多少个有效 token？(2) Padding 模式下，所有 expert 合计的有效 token 数 / 处理的 token 数是多少？(3) 用 varlen 替代 padding 后，节省的 attention 计算量是多少倍？

```python
B, S, E, k = 4, 8192, 64, 1
total_tokens = B * S
# TODO: 补全三行
# tokens_per_expert = ?
# efficiency = ?       # 有效 / 总处理
# speedup_varlen = ?   # varlen 相对 padding 的加速比

assert tokens_per_expert == 512, f"tokens_per_expert 应该是 512，你得到 {tokens_per_expert}"
assert abs(efficiency - 1/64) < 1e-6, f"efficiency 应该是 1/64"
assert abs(speedup_varlen - 64) < 1e-6, f"speedup_varlen 应该是 64"
print("作业 3 通过：理解 MoE padding 浪费")
```

小提示：每个 token 被路由到 $k$ 个 expert，所以每个 expert 平均收到 $B \cdot S \cdot k / E$ 个 token。Padding 模式下 expert 仍然处理完整 $B \cdot S$ 张量，效率就是两者之比。Varlen 把有效 token 拼起来算，加速比是效率的倒数。
